## Prof. Alfio Ferrara
# Introduzione all'apprendimento automatico
### Tecnologie dei dati e del linguaggio

## Apprendimento non supervisionato

Nei problemi di apprendimento supervisionato, ogni esempio del dataset è accompagnato 
da un'etichetta: sappiamo già a quale categoria appartiene, e il modello impara a 
riprodurre quella classificazione. Nell'apprendimento **non supervisionato** questa 
informazione non esiste. Il modello riceve soltanto i dati grezzi e ha il compito di 
scoprirne la struttura nascosta autonomamente.

Questo tipo di apprendimento risponde a una domanda diversa: non *"a quale categoria 
appartiene questo esempio?"*, ma *"esistono gruppi naturali in questi dati? E se sì, 
quali esempi si assomigliano?"*

Una delle tecniche più comuni è il **clustering**: l'algoritmo raggruppa gli esempi in 
modo che quelli simili finiscano nello stesso gruppo, e quelli diversi in gruppi separati. 
La nozione di *simile* è formale: dipende da come rappresentiamo i dati come vettori e 
da quale misura di distanza usiamo.

Nel nostro caso, ogni ricetta diventerà un vettore numerico costruito a partire dal testo 
della sua procedura. L'obiettivo è verificare se un algoritmo di clustering riesce a 
raggruppare ricette affini **senza mai aver visto le loro categorie**.

Esistono diversi algoritmi di clustering, ciascuno con assunzioni diverse sulla forma 
dei gruppi e sul numero di cluster atteso. In questa sezione useremo il 
**clustering gerarchico agglomerativo**: un approccio che non richiede di fissare 
in anticipo il numero di gruppi, ma costruisce progressivamente una gerarchia di 
similarità tra gli esempi, rappresentata visivamente da un **dendrogramma**.

In [ ]:
import pandas as pd
import ast

In [ ]:
# Load dataset
df = pd.read_csv('/Users/Flint/Data/recipes/it_recipes.csv', index_col=0)
df.dropna(inplace=True)

# Parse ingredients from string to list
df['Ingredienti'] = df['Ingredienti'].apply(ast.literal_eval)

# Isolate the text column we will work with
texts = df['Steps'].tolist()
names = df['Nome'].tolist()

# Quick inspection
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nCategories found: {df['Categoria'].unique()}")
print(f"\n--- Example: first recipe ---")
print(f"Name: {names[0]}")
print(f"Text preview: {texts[0][:100]}...")

### Rappresentare il testo come vettore: Term Frequency

Per trasformare il testo di una ricetta in un vettore numerico, partiamo dal conteggio 
delle parole. Per ogni termine $t$ presente in un documento $d$, definiamo la sua 
**Term Frequency normalizzata** come la proporzione di volte in cui compare rispetto 
al totale delle occorrenze nel documento:

$$tf(t, d) = \frac{count(t \in d)}{\sum_{i=1}^{n} count(t_i \in d)}$$

Normalizzare il conteggio ci permette di confrontare documenti di lunghezza diversa 
su una scala comune. Una parola che compare 10 volte in un testo di 100 token ha 
lo stesso peso di una che compare 20 volte in un testo di 200 token.

Per suddividere il testo in token useremo il tokenizzatore del modello **BERT 
addestrato sull'italiano**: rispetto a una semplice divisione per spazi, questo 
tokenizzatore gestisce meglio le varianti morfologiche della lingua italiana 
(suffissi, forme flesse) e produce una rappresentazione più consistente del vocabolario.

In [ ]:
from transformers import AutoTokenizer, logging as hf_logging
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import re 
import nltk
from nltk.tokenize import word_tokenize

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('dbmdz/bert-base-italian-cased')
hf_logging.set_verbosity_error()

def bert_tokenize(text):
    """Tokenize text using Italian BERT tokenizer, removing special tokens."""
    tokens = bert_tokenizer.tokenize(text)
    # Remove BERT special subword prefix and keep only full tokens
    tokens = [t for t in tokens if not t.startswith("##")]
    # Keep only tokens with at least one alphabetic character (including some accented)
    tokens = [t for t in tokens if re.search(r'[a-zA-Zàèéìíîòóùú]', t)]
    return tokens

def nltk_tokenize(text):
    """Tokenize text using NLTK word_tokenize.
    Removes punctuation, numbers, and spurious tokens.
    Keeps only tokens containing at least one Italian alphabetic character.
    """
    tokens = word_tokenize(text, language='italian')
    tokens = [t.lower() for t in tokens]
    # Keep only tokens with at least one alphabetic character (including accented)
    tokens = [t for t in tokens if re.search(r'[a-zA-ZàèéìíîòóùúÀÈÉÌÍÎÒÓÙÚ]', t)]
    # Remove tokens that are only one character long
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

tokenizer = nltk_tokenize

# Build count matrix
count_vectorizer = CountVectorizer(tokenizer=tokenizer, lowercase=True, token_pattern=None, max_features=2000)
count_matrix = count_vectorizer.fit_transform(texts)

# Compute normalized TF
count_array = count_matrix.toarray().astype(float)
row_sums = count_array.sum(axis=1, keepdims=True)
tf_matrix = count_array / row_sums

# Inspect the result
vocab = count_vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")
print(f"TF matrix shape: {tf_matrix.shape}")

X = pd.DataFrame(tf_matrix, columns=vocab)
index2title = {i: name for i, name in enumerate(names)}

Visualiziamo i primi 10 token più importanti della prima ricetta

In [ ]:
X.iloc[0].sort_values(ascending=False).head(10)

#### Visualizzare dati ad alta dimensione: t-SNE

La matrice $X$ che abbiamo costruito ha tante colonne quante sono le parole del 
vocabolario — potenzialmente migliaia. Ogni ricetta è un punto in uno spazio 
ad altissima dimensione, impossibile da visualizzare direttamente.

Per rappresentare questi dati in due dimensioni usiamo **t-SNE** (*t-distributed 
Stochastic Neighbor Embedding*): un algoritmo che proietta i punti in 2D cercando 
di preservare le relazioni di vicinanza originali.

Il procedimento avviene in due fasi. Prima, nello spazio originale ad alta dimensione, 
t-SNE misura per ogni punto quanto sono vicini tutti gli altri, costruendo una 
distribuzione di probabilità: i vicini ricevono probabilità alta, i punti lontani 
probabilità bassa. Poi inizializza una proiezione casuale in 2D e, iterazione dopo 
iterazione, sposta i punti nel piano finché le probabilità di vicinanza nel 2D 
assomigliano il più possibile a quelle originali. Il risultato è una disposizione 
in cui due ricette con vettori simili tendono a restare vicine anche nella proiezione.

È importante sottolineare che t-SNE è uno strumento **esclusivamente visivo**: 
le distanze assolute tra punti lontani non sono interpretabili, e la forma dei 
gruppi può variare a seconda dei parametri scelti. Quello che possiamo leggere 
con fiducia è la presenza di **aggregazioni locali**: insiemi di punti che 
tendono a stare insieme nel piano 2D riflettono ricette che il modello considera simili.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
X_2d = tsne.fit_transform(X.values)

limit = 50
X_2d_small = X_2d[:limit]

# Build a dataframe for seaborn
plot_df = pd.DataFrame({
    'x': X_2d_small[:, 0],
    'y': X_2d_small[:, 1],
    'title': [index2title[i] for i in range(limit)]
})

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=plot_df, x='x', y='y', ax=ax, alpha=0.7)

for _, row in plot_df.iterrows():
    ax.annotate(row['title'], (row['x'], row['y']),
                fontsize=8, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')

ax.set_title('Ricette nello spazio 2D (t-SNE) — rappresentazione TF')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
sns.despine()
plt.tight_layout()
plt.show()

### Clustering gerarchico agglomerativo

L'idea alla base del **clustering gerarchico agglomerativo** è semplice: partiamo 
dal caso estremo in cui ogni ricetta è un gruppo a sé, e poi procediamo per fusioni 
successive. Ad ogni passo, i due gruppi più simili vengono uniti in uno solo. 
Si continua finché non rimane un unico gruppo che contiene tutte le ricette.

Il risultato non è una singola partizione dei dati, ma una **gerarchia completa di 
raggruppamenti**, dal più specifico (ogni ricetta da sola) al più generale (tutte 
le ricette insieme). Questo è il vantaggio principale rispetto ad altri algoritmi 
di clustering: non dobbiamo decidere in anticipo quanti gruppi vogliamo — la struttura 
emerge dai dati stessi, e possiamo osservarla a diversi livelli di granularità.

Per misurare la similarità tra gruppi — non più tra singoli punti — dobbiamo definire 
un criterio di **linkage**, cioè una regola che stabilisce la distanza tra due insiemi 
di punti. Useremo il **linkage di Ward**, che ad ogni fusione sceglie di unire i due 
gruppi che minimizzano l'aumento della varianza interna: tende a produrre cluster 
compatti e di dimensione equilibrata.

Questa gerarchia di fusioni si rappresenta visivamente con un **dendrogramma**: 
un diagramma ad albero in cui le foglie sono le singole ricette e ogni nodo interno 
rappresenta una fusione. L'altezza a cui due rami si uniscono indica la distanza 
a cui quella fusione è avvenuta — fusioni basse uniscono ricette molto simili, 
fusioni alte uniscono gruppi già abbastanza distanti. Tagliando il dendrogramma 
ad una certa altezza otteniamo una partizione del dataset in un numero di cluster 
che possiamo scegliere *a posteriori*, dopo aver osservato la struttura.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
limit = 50

# Apply limit
X_cluster = X.iloc[:limit] if limit else X
titles_cluster = [index2title[i] for i in range(len(X_cluster))]

# Compute hierarchical clustering with Ward linkage
Z = linkage(X_cluster.values, method='ward')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z,
    labels=titles_cluster,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax
)

ax.set_title('Dendrogramma — clustering gerarchico agglomerativo (TF)')
ax.set_xlabel('Ricette')
ax.set_ylabel('Distanza (Ward)')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster

In [ ]:
cut_threshold = 0.15

# Assign cluster labels
cluster_labels = fcluster(Z, t=cut_threshold, criterion='distance')

# Build cluster dictionary: {cluster_id: [recipe titles]}
clusters = {}
for idx, cluster_id in enumerate(cluster_labels):
    clusters.setdefault(cluster_id, []).append(titles_cluster[idx])

# Inspect clusters
print(f"Number of clusters at threshold {cut_threshold}: {len(clusters)}\n")
for cluster_id, titles in sorted(clusters.items()):
    print(f"Cluster {cluster_id} ({len(titles)} ricette):")
    for title in titles:
        print(f"  - {title}")
    print()

### Dal conteggio al peso: TF-IDF

Osservando i termini che dominano i nostri vettori TF, emerge un problema chiaro: 
le parole più frequenti non sono le più *informative*. Termini come "aggiungete", 
"versate", "mescolate" compaiono in quasi tutte le ricette — hanno una frequenza 
alta, ma ci dicono poco su cosa distingue una ricetta dall'altra.

Il problema non riguarda solo le parole grammaticali come articoli e preposizioni: 
esiste una classe di termini specifici del dominio culinario che sono ugualmente 
poco informativi proprio perché **troppo diffusi**. Per catturare questa idea 
introduciamo il concetto di **Document Frequency**.

#### Document Frequency (DF)

La *document frequency* di un termine $t$ è semplicemente il numero di documenti 
del corpus in cui $t$ compare almeno una volta:

$$df(t) = |\{d \in D : t \in d\}|$$

Un termine con $df$ alta è presente in molti documenti: è poco selettivo, e quindi 
poco utile per distinguere un documento dall'altro. Un termine con $df$ bassa è 
raro nel corpus: quando compare, porta informazione specifica.

#### Inverse Document Frequency (IDF)

Per trasformare questa intuizione in un peso numerico, definiamo l'**Inverse Document 
Frequency**: l'inverso della document frequency, normalizzato sul numero totale di 
documenti $|D|$:

$$idf(t) = \frac{|D|}{df(t)}$$

Più un termine è diffuso, più il suo IDF si avvicina a 1. Più è raro, più IDF cresce. 
Il problema di questa formulazione è che la crescita è troppo ripida: un termine 
presente in un solo documento su mille avrebbe un peso mille volte superiore a uno 
presente ovunque. Questa sproporzione renderebbe i vettori dominati da termini 
rarissimi, magari errori di battitura o parole hapax, che non portano informazione 
utile.

Per comprimere questa dinamica si applica il **logaritmo**:

$$idf(t) = \log\frac{|D|}{df(t)}$$

Il logaritmo preserva l'ordinamento — i termini rari pesano comunque più di quelli 
comuni — ma attenua le differenze estreme, rendendo la scala più equilibrata. 
Lo si vede chiaramente confrontando l'andamento delle due curve al variare di $df$.

#### TF-IDF

Combinando le due componenti otteniamo il peso **TF-IDF**: per ogni termine $t$ 
in ogni documento $d$, il peso è il prodotto della sua frequenza nel documento 
per la sua rarità nel corpus:

$$tfidf(t, d) = tf(t, d) \times idf(t)$$

Un termine ottiene peso alto solo se è frequente *in quel documento* **e** raro 
*nel corpus generale*: esattamente la proprietà che vogliamo per distinguere 
le ricette tra loro.

In [ ]:
# Number of documents
N = len(X)

# Compute document frequency: number of documents where each term appears at least once
df_values = (X.values > 0).sum(axis=0)

# IDF without log
idf_raw = N / df_values

# IDF with log
idf_log = np.log(N / df_values)

# Sort both by descending IDF (raw)
sorted_indices = np.argsort(idf_raw)[::-1]
idf_raw_sorted = idf_raw[sorted_indices]
idf_log_sorted = idf_log[sorted_indices]
x_positions = np.arange(len(sorted_indices))

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: IDF without log
sns.lineplot(x=x_positions, y=idf_raw_sorted, ax=axes[0], linewidth=1)
axes[0].set_title('IDF senza logaritmo')
axes[0].set_xlabel('Termini (ordinati per IDF decrescente)')
axes[0].set_ylabel('IDF')

# Right: IDF with log
sns.lineplot(x=x_positions, y=idf_log_sorted, ax=axes[1], linewidth=1)
axes[1].set_title('IDF con logaritmo')
axes[1].set_xlabel('Termini (ordinati per IDF decrescente)')
axes[1].set_ylabel('IDF (log)')

sns.despine()
plt.tight_layout()
plt.show()

# Print the 10 lowest IDF terms (most diffuse) and 10 highest (most specific)
vocab_array = np.array(vocab)
print("--- 10 termini con IDF più basso (più diffusi nel corpus) ---")
for term, val in zip(vocab_array[sorted_indices[-10:]], idf_log_sorted[-10:]):
    print(f"  {term:<25} idf={val:.3f}")

print("\n--- 10 termini con IDF più alto (più specifici) ---")
for term, val in zip(vocab_array[sorted_indices[:10]], idf_log_sorted[:10]):
    print(f"  {term:<25} idf={val:.3f}")

### Applichiamo Tf-Idf come criterio di peso dei token nella vettorizzazione

In [ ]:
tfidf_matrix = tf_matrix * idf_log
X_tfidf = pd.DataFrame(tfidf_matrix, columns=vocab)

Parole più rilevanti della prima ricetta col nuovo criterio

In [ ]:
X_tfidf.iloc[0].sort_values(ascending=False).head(10)

### Applichiamo la procedura di clustering ai nuovi vettori

In [ ]:
limit = 50

# Apply limit
X_cluster = X_tfidf.iloc[:limit] if limit else X_tfidf
titles_cluster = [index2title[i] for i in range(len(X_cluster))]

# --- t-SNE ---
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
X_2d = tsne.fit_transform(X_cluster.values)

plot_df = pd.DataFrame({
    'x': X_2d[:, 0],
    'y': X_2d[:, 1],
    'title': titles_cluster
})

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=plot_df, x='x', y='y', ax=ax, alpha=0.7)
for _, row in plot_df.iterrows():
    ax.annotate(row['title'], (row['x'], row['y']),
                fontsize=8, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')
ax.set_title('Ricette nello spazio 2D (t-SNE) — rappresentazione TF-IDF')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
sns.despine()
plt.tight_layout()
plt.show()

# --- Hierarchical clustering ---
Z = linkage(X_cluster.values, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, labels=titles_cluster, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title('Dendrogramma — clustering gerarchico agglomerativo (TF-IDF)')
ax.set_xlabel('Ricette')
ax.set_ylabel('Distanza (Ward)')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
cut_threshold = 0.38


# --- Flat clusters ---
cluster_labels = fcluster(Z, t=cut_threshold, criterion='distance')

clusters = {}
for idx, cluster_id in enumerate(cluster_labels):
    clusters.setdefault(cluster_id, []).append(titles_cluster[idx])

print(f"Numero di cluster alla soglia {cut_threshold}: {len(clusters)}\n")
for cluster_id, titles in sorted(clusters.items()):
    print(f"Cluster {cluster_id} ({len(titles)} ricette):")
    print(", ".join(titles))
    print()

In [ ]:
# --- t-SNE colored by cluster ---
plot_df['cluster'] = cluster_labels.astype(str)

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=plot_df, x='x', y='y', hue='cluster',
                palette='tab10', ax=ax, alpha=0.8, s=80)
for _, row in plot_df.iterrows():
    ax.annotate(row['title'], (row['x'], row['y']),
                fontsize=8, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')
ax.set_title('Ricette nello spazio 2D (t-SNE) — cluster TF-IDF')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()

### Esempio completo

In [ ]:
# Compute hierarchical clustering on full TF-IDF matrix
Z_full = linkage(X_tfidf.values, method='ward')
cluster_labels_full = fcluster(Z_full, t=cut_threshold, criterion='distance')

# t-SNE on full dataset
tsne_full = TSNE(n_components=2, random_state=42, perplexity=30)
X_2d_full = tsne_full.fit_transform(X_tfidf.values)

plot_df_full = pd.DataFrame({
    'x': X_2d_full[:, 0],
    'y': X_2d_full[:, 1],
    'cluster': cluster_labels_full.astype(str)
})

# Add recipe titles to full dataframe
titles_full = [index2title[i] for i in range(len(X_tfidf))]
plot_df_full['title'] = titles_full

# --- Cluster selection ---
# Set to None to show all clusters, or specify a list of cluster ids to filter
selected_clusters = ['1', '2', '3']

# Filter if selection is active
if selected_clusters:
    mask = plot_df_full['cluster'].isin(selected_clusters)
    plot_df_viz = plot_df_full[mask].copy()
else:
    plot_df_viz = plot_df_full.copy()

fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=plot_df_viz, x='x', y='y', hue='cluster',
                palette='tab10', ax=ax, alpha=0.7, s=40)
ax.set_title('Tutte le ricette — cluster TF-IDF (t-SNE)')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()

# --- Print recipe titles for selected clusters ---
clusters_to_print = selected_clusters if selected_clusters else sorted(plot_df_full['cluster'].unique())
for cluster_id in clusters_to_print:
    titles_in_cluster = plot_df_full[plot_df_full['cluster'] == cluster_id]['title'].tolist()
    print(f"\nCluster {cluster_id} ({len(titles_in_cluster)} ricette):")
    for title in titles_in_cluster:
        print(f"  - {title}")

## Apprendimento per rinforzo

Nel paradigma che abbiamo visto finora, il modello riceve dati e impara da essi. 
Nell'**apprendimento per rinforzo** la struttura è 
diversa: non esiste un dataset di partenza. Esiste un **agente** che interagisce 
con un **ambiente**, compie **azioni** e riceve in risposta un **reward** — un segnale 
numerico che indica quanto quella scelta è stata vantaggiosa.

L'obiettivo dell'agente non è massimizzare il reward di una singola azione, ma 
imparare una **politica** — una strategia che, nel corso di molte interazioni, 
massimizza il reward *cumulativo* nel tempo. Questo implica che l'agente debba 
talvolta accettare una scelta subottimale nel breve periodo per ottenere un 
vantaggio maggiore in seguito.

Il ciclo fondamentale è sempre lo stesso: l'agente osserva lo **stato** corrente 
dell'ambiente, sceglie un'azione, riceve un reward e osserva il nuovo stato. 
Questo ciclo si ripete per molti **episodi**, e ad ogni iterazione l'agente 
aggiorna la propria conoscenza.

Esistono molti algoritmi di apprendimento per rinforzo, che si differenziano 
principalmente per come rappresentano e aggiornano la conoscenza acquisita. 
Alcuni lavorano direttamente sulla politica, cercando di migliorarla passo dopo 
passo. Altri stimano il valore atteso del reward futuro per ogni coppia 
stato-azione, e derivano la politica da questa stima.

Noi useremo **Q-learning**, che appartiene a questa seconda famiglia. L'idea 
centrale è costruire una **Q-table**: una tabella in cui ogni riga è uno stato 
possibile dell'ambiente e ogni colonna è un'azione disponibile. Ogni cella 
contiene il valore $Q(s, a)$, la stima del reward cumulativo atteso se 
l'agente si trova nello stato $s$ e compie l'azione $a$, agendo poi in modo 
ottimale.

La tabella viene aggiornata ad ogni step secondo la regola:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Il termine tra parentesi si chiama **errore TD** (*Temporal Difference*): è la 
differenza tra il valore stimato attuale $Q(s, a)$ e una stima migliore costruita 
usando il reward appena ricevuto $r$ e il valore massimo raggiungibile dallo stato 
successivo $s'$. Il parametro $\alpha$ controlla la velocità di apprendimento, 
mentre $\gamma$, detto **fattore di sconto**, bilancia l'importanza del reward 
immediato rispetto a quello futuro.

Scegliamo Q-learning perché la sua struttura tabulare rende il processo di 
apprendimento completamente trasparente: potremo osservare la Q-table prima, 
durante e dopo il training, e leggere direttamente cosa ha imparato l'agente.

### Il Cuoco: costruire un menu con l'apprendimento per rinforzo

Per esplorare il Q-learning costruiamo un esempio concreto: un agente — il nostro 
**Cuoco** — deve imparare a comporre un menu completo scegliendo un piatto per 
ciascuna delle quattro portate tradizionali: antipasto, primo, secondo e dolce.

All'inizio il Cuoco non sa nulla di cucina. Ad ogni portata sceglie un piatto 
a caso dalla lista disponibile, e riceve un feedback immediato: ha scelto qualcosa 
di appropriato per quella portata, oppure no? Nel tempo, accumulando esperienza 
su molti menu, impara ad associare ogni portata ai piatti più adatti.

**Gli stati** del gioco sono le quattro portate, affrontate in sequenza:

$$antipasto \rightarrow primo \rightarrow secondo \rightarrow dolce$$

Ma lo stato non è solo la portata corrente: include anche quali ingredienti principali 
sono già stati usati nelle portate precedenti. Questo perché le scelte non sono 
indipendenti — scegliere un piatto a base di pasta come primo esclude di fatto 
piatti simili nelle portate successive, penalizzando la varietà del menu. 
Lo stato è quindi la coppia *(portata corrente, insieme degli ingredienti già usati)*.

**Le azioni** disponibili sono sempre le stesse dieci, indipendentemente dalla 
portata corrente:

| | Piatto | Ingrediente principale |
|---|---|---|
| 0 | Bruschetta | pane |
| 1 | Pasta al pomodoro | pasta |
| 2 | Risotto ai funghi | riso |
| 3 | Bistecca alla fiorentina | carne |
| 4 | Salmone al forno | pesce |
| 5 | Insalata caprese | pomodoro |
| 6 | Tiramisù | mascarpone |
| 7 | Panna cotta | panna |
| 8 | Minestrone | verdura |
| 9 | Frittata | uova |

Il **reward** che il Cuoco riceve ad ogni scelta combina due componenti: 
l'appropriatezza del piatto per quella portata — scegliere il Tiramisù come 
antipasto porta un feedback negativo, sceglierlo come dolce porta un feedback 
positivo — e una **penalità** se l'ingrediente principale del piatto scelto 
è già stato usato in una portata precedente. Questo crea una tensione reale 
tra le scelte: un piatto ottimo per la portata corrente potrebbe rivelarsi 
una scelta miope se occupa un ingrediente che sarebbe stato prezioso più avanti.

La **Q-table** che l'agente costruisce ha una riga per ogni stato possibile — 
ogni combinazione di portata e insieme di ingredienti già usati — e una colonna 
per ciascuna delle dieci azioni disponibili. Le righe raccontano storie diverse: 
il valore di scegliere la Pasta al pomodoro come primo cambia a seconda di cosa 
è già stato servito come antipasto. È proprio questa dipendenza tra le scelte 
che rende il problema interessante e giustifica l'uso del Q-learning.

In [ ]:
# --- Environment definition ---

COURSES = ['Antipasto', 'Primo', 'Secondo', 'Dolce']
DISHES = [
    'Bruschetta',        # 0
    'Pasta al pomodoro', # 1
    'Risotto ai funghi', # 2
    'Bistecca',          # 3
    'Salmone al forno',  # 4
    'Insalata caprese',  # 5
    'Tiramisù',          # 6
    'Panna cotta',       # 7
    'Minestrone',        # 8
    'Frittata'           # 9
]

N_COURSES = len(COURSES)
N_ACTIONS = len(DISHES)

# Reward table: rows = courses, columns = dishes
# Values reflect culinary appropriateness: +1 good, 0 neutral, -1 wrong
REWARD_TABLE = np.array([
#   Bru  Pom  Ris  Bis  Sal  Cap  Tir  Pan  Min  Fri
    [ 1,   0,   0,  -1,   0,   1,  -1,  -1,   0,   1],  # Antipasto
    [-1,   1,   1,   0,   0,  -1,  -1,  -1,   1,   0],  # Primo
    [-1,  -1,  -1,   1,   1,   0,  -1,  -1,   0,   1],  # Secondo
    [-1,  -1,  -1,  -1,  -1,  -1,   1,   1,  -1,  -1],  # Dolce
], dtype=float)

# Main ingredient per dish — repeated ingredients across courses are penalized
MAIN_INGREDIENTS = {
    0: 'pane',      # Bruschetta
    1: 'pasta',     # Pasta al pomodoro
    2: 'riso',      # Risotto ai funghi
    3: 'carne',     # Bistecca
    4: 'pesce',     # Salmone al forno
    5: 'verdura',   # Insalata caprese
    6: 'uova',      # Tiramisù
    7: 'panna',     # Panna cotta
    8: 'verdura',   # Minestrone
    9: 'uova'       # Frittata
}

# Penalty applied when a chosen dish shares its main ingredient with a previous course
REPETITION_PENALTY = -2.0

def make_state(course_idx, used_ingredients):
    """Build a hashable state from course index and set of used ingredients."""
    return (course_idx, frozenset(used_ingredients))

def get_reward(course_idx, action, used_ingredients):
    """Return the reward for choosing action at the current course.
    Applies a repetition penalty if the dish's main ingredient was already used.
    """
    reward = REWARD_TABLE[course_idx, action]
    if MAIN_INGREDIENTS[action] in used_ingredients:
        reward += REPETITION_PENALTY
    return reward

def is_terminal(course_idx):
    """Return True if the episode is over (all courses assigned)."""
    return course_idx >= N_COURSES

# --- Visualization helpers ---

def plot_reward_table():
    """Display the full reward structure:
    1. base appropriateness (course x dish)
    2. repetition penalty (dish x dish) based on shared main ingredient
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # Left: base appropriateness table
    df_reward = pd.DataFrame(REWARD_TABLE, index=COURSES, columns=DISHES)
    sns.heatmap(df_reward, annot=True, fmt='.0f', cmap='RdYlGn',
                center=0, linewidths=0.5, ax=axes[0])
    axes[0].set_title('Appropriatezza portata/piatto')
    axes[0].set_xlabel('Piatto')
    axes[0].set_ylabel('Portata')
    axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

    # Right: dish x dish repetition penalty matrix
    # cell (i,j) = REPETITION_PENALTY if dishes i and j share the same main ingredient
    penalty_matrix = np.zeros((N_ACTIONS, N_ACTIONS))
    for i in range(N_ACTIONS):
        for j in range(N_ACTIONS):
            if i != j and MAIN_INGREDIENTS[i] == MAIN_INGREDIENTS[j]:
                penalty_matrix[i, j] = REPETITION_PENALTY

    df_penalty = pd.DataFrame(penalty_matrix, index=DISHES, columns=DISHES)
    sns.heatmap(df_penalty, annot=True, fmt='.0f', cmap='RdYlGn',
                center=0, linewidths=0.5, ax=axes[1])
    axes[1].set_title(f'Penalità per ripetizione ingrediente tra portate')
    axes[1].set_xlabel('Piatto')
    axes[1].set_ylabel('Piatto')
    axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    plt.suptitle('Struttura completa del reward', fontsize=13)
    sns.despine()
    plt.tight_layout()
    plt.show()
    
def plot_q_table(Q, title='Q-table'):
    """Display Q-table as a heatmap, averaged over all states per course.
    Since the state space is now (course, used_ingredients), we show
    the mean Q-values for each course across all visited states.
    """
    # Aggregate Q-values by course index
    course_q = {i: [] for i in range(N_COURSES)}
    for (course_idx, _), q_values in Q.items():
        course_q[course_idx].append(q_values)

    # Build mean Q matrix: rows = courses, columns = dishes
    mean_q = np.zeros((N_COURSES, N_ACTIONS))
    for i in range(N_COURSES):
        if course_q[i]:
            mean_q[i] = np.mean(course_q[i], axis=0)

    df_q = pd.DataFrame(mean_q, index=COURSES, columns=DISHES)
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df_q, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, ax=ax)
    ax.set_title(f'{title}  (Q-values medi per portata)')
    ax.set_xlabel('Piatto')
    ax.set_ylabel('Portata')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()

def plot_episode_rewards(rewards, window=50):
    """Plot total reward per episode with a rolling average."""
    df_r = pd.DataFrame({'reward': rewards})
    df_r['rolling'] = df_r['reward'].rolling(window=window, min_periods=1).mean()
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.lineplot(data=df_r, x=df_r.index, y='reward',
                 alpha=0.3, color='steelblue', ax=ax, label='reward per episodio')
    sns.lineplot(data=df_r, x=df_r.index, y='rolling',
                 color='steelblue', ax=ax, label=f'media mobile ({window} ep.)')
    ax.set_title('Reward totale per episodio')
    ax.set_xlabel('Episodio')
    ax.set_ylabel('Reward totale')
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.show()

plot_reward_table()

In [ ]:
# --- Agent ---

class CookAgent:
    def __init__(self, n_actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        """
        Q-learning agent with dictionary-based Q-table to support
        the expanded state space (course_idx, used_ingredients).

        alpha:         learning rate
        gamma:         discount factor
        epsilon:       initial exploration rate
        epsilon_decay: multiplicative decay applied after each episode
        epsilon_min:   minimum exploration rate
        """
        self.n_actions     = n_actions
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min   = epsilon_min

        # Q-table as dictionary: state -> np.array of shape (n_actions,)
        self.Q = {}

    def _get_q(self, state):
        """Return Q-values for a state, initializing to zeros if unseen."""
        if state not in self.Q:
            self.Q[state] = np.zeros(self.n_actions)
        return self.Q[state]

    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)    # explore
        return np.argmax(self._get_q(state))            # exploit

    def update(self, state, action, reward, next_state, terminal):
        """Apply the Q-learning update rule."""
        if terminal:
            td_target = reward
        else:
            td_target = reward + self.gamma * np.max(self._get_q(next_state))

        q_values         = self._get_q(state)
        td_error         = td_target - q_values[action]
        q_values[action] += self.alpha * td_error

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def copy_q(self):
        """Return a deep copy of the current Q-table."""
        return {state: q.copy() for state, q in self.Q.items()}


# --- Training loop ---

def train(agent, n_episodes=2000, snapshot_episodes=None):
    """
    Train the agent for n_episodes episodes.
    snapshot_episodes: list of episode numbers at which to save a Q-table snapshot.
    Returns episode rewards and Q-table snapshots.
    """
    if snapshot_episodes is None:
        snapshot_episodes = [1, 10, 50, 100, 500, n_episodes]

    episode_rewards = [0]
    snapshots       = {0: agent.copy_q()}

    for episode in range(1, n_episodes + 1):
        course_idx       = 0
        used_ingredients = set()
        total_reward     = 0

        while not is_terminal(course_idx):
            state      = make_state(course_idx, used_ingredients)
            action     = agent.choose_action(state)
            reward     = get_reward(course_idx, action, used_ingredients)
            next_idx   = course_idx + 1
            terminal   = is_terminal(next_idx)
            next_state = make_state(next_idx, used_ingredients | {MAIN_INGREDIENTS[action]})

            agent.update(state, action, reward, next_state, terminal)

            used_ingredients.add(MAIN_INGREDIENTS[action])
            total_reward += reward
            course_idx = next_idx

        agent.decay_epsilon()
        episode_rewards.append(total_reward)

        if episode in snapshot_episodes:
            snapshots[episode] = agent.copy_q()

    return episode_rewards, snapshots


# --- Run ---
N_EPISODES = 2000

agent = CookAgent(
    n_actions     = N_ACTIONS,
    alpha         = 0.1,
    gamma         = 0.9,
    epsilon       = 1.0,
    epsilon_decay = 0.995,
    epsilon_min   = 0.01
)

episode_rewards, snapshots = train(agent, n_episodes=N_EPISODES)

### Prima dell'apprendimento

In [ ]:
plot_q_table(snapshots[0], title='Q-table — prima del training (episodio 0)')

### Processo di apprendimento

In [ ]:
from IPython.display import clear_output
import time 

In [ ]:
plot_episode_rewards(episode_rewards, window=50)

In [ ]:
for episode in [0, 1, 10, 50, 100, 500, N_EPISODES]:
    clear_output(wait=True)
    plot_q_table(snapshots[episode], title=f'Q-table — episodio {episode}')
    time.sleep(1)


### Il cuoco all'opera

In [ ]:
def serve_meal(agent, delay=1.5):
    """
    Use the trained agent to compose a full meal step by step using greedy policy.
    Prints each choice with a delay to simulate a narrative animation.
    """
    course_idx       = 0
    used_ingredients = set()
    menu             = []
    total_reward     = 0

    print("=" * 55)
    print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
    print("=" * 55)
    time.sleep(delay)

    while not is_terminal(course_idx):
        state    = make_state(course_idx, used_ingredients)
        q_values = agent._get_q(state)
        action   = np.argmax(q_values)
        reward   = get_reward(course_idx, action, used_ingredients)

        # Show deliberation
        clear_output(wait=True)
        print("=" * 55)
        print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
        print("=" * 55)
        for past_course, past_action in zip(COURSES[:course_idx], menu):
            print(f"\n  {past_course:<12} →  {DISHES[past_action]}")
        print(f"\n  {COURSES[course_idx]:<12} →  ...")
        print(f"\n  Il Cuoco considera:")
        for i, (dish, q) in enumerate(zip(DISHES, q_values)):
            marker = " ← scelta" if i == action else ""
            print(f"  {dish:<25} Q={q:+.2f}{marker}")

        time.sleep(delay)

        # Show choice
        clear_output(wait=True)
        print("=" * 55)
        print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
        print("=" * 55)
        for past_course, past_action in zip(COURSES[:course_idx], menu):
            print(f"\n  {past_course:<12} →  {DISHES[past_action]}")
        print(f"\n  {COURSES[course_idx]:<12} →  {DISHES[action]}  (reward: {reward:+.1f})")

        used_ingredients.add(MAIN_INGREDIENTS[action])
        menu.append(action)
        total_reward += reward
        course_idx += 1

        time.sleep(delay)

    # Final summary
    clear_output(wait=True)
    print("=" * 55)
    print("       🍽️  IL MENU È PRONTO  🍽️")
    print("=" * 55)
    for course, action in zip(COURSES, menu):
        print(f"\n  {course:<12} →  {DISHES[action]}")
    print(f"\n  Reward totale: {total_reward:+.1f}")
    print("=" * 55)

serve_meal(agent, delay=1.5)

### Esplorazione e temperatura

Un agente completamente greedy sceglie sempre il piatto con Q-value più alto — 
ma questo lo rende prevedibile e rigido. Esiste un modo più sfumato di usare 
la Q-table: invece di prendere sempre il massimo, possiamo **campionare** un'azione 
usando i Q-value come base per una distribuzione di probabilità.

Lo strumento è la funzione **softmax**, parametrizzata da un valore $\tau$ detto 
**temperatura**:

$$P(a | s) = \frac{e^{Q(s,a) / \tau}}{\sum_{a'} e^{Q(s,a') / \tau}}$$

La temperatura controlla quanto la distribuzione è concentrata:
- con $\tau$ **bassa** (es. 0.1) il softmax amplifica le differenze — la probabilità 
si concentra quasi tutta sull'azione migliore, comportamento simile al greedy
- con $\tau$ **alta** (es. 5.0) il softmax appiattisce le differenze — tutte le 
azioni diventano quasi equiprobabili, comportamento simile alla scelta casuale
- con $\tau = 1.0$ i Q-value vengono usati direttamente come logit

Questo meccanismo ci permette di regolare quanta **creatività** vogliamo nel menu: 
un Cuoco con temperatura alta sorprenderà di più, uno con temperatura bassa sarà 
più affidabile.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey='row')

scenarios = [
    # (course_idx, used_ingredients, description)
    (0, set(),           'Antipasto — nessun ingrediente usato'),
    (1, {'verdura'},     'Primo — verdura già usata (Insalata caprese come antipasto)'),
]

for row, (course_idx, used_ingredients, description) in enumerate(scenarios):
    state    = make_state(course_idx, used_ingredients)
    q_values = agent._get_q(state)

    for col, tau in enumerate(temperatures):
        ax    = axes[row, col]
        probs = softmax(q_values, temperature=tau)

        df_plot = pd.DataFrame({
            'piatto':      DISHES,
            'probabilità': probs,
            'penalizzato': [MAIN_INGREDIENTS[i] in used_ingredients 
                           for i in range(N_ACTIONS)]
        }).sort_values('probabilità', ascending=False)

        # Color penalized dishes differently
        palette = ['tomato' if p else 'steelblue' for p in df_plot['penalizzato']]
        sns.barplot(data=df_plot, x='probabilità', y='piatto',
                    hue='piatto', palette=palette, legend=False, ax=ax)
        ax.set_title(f'{description}\ntemperatura τ = {tau}')
        ax.set_xlabel('Probabilità')
        ax.set_ylabel('Piatto')
        ax.set_xlim(0, 1)

        for i, (_, row_data) in enumerate(df_plot.iterrows()):
            ax.text(row_data['probabilità'] + 0.01, i,
                    f"{row_data['probabilità']:.2f}",
                    va='center', fontsize=9)

sns.despine()
plt.suptitle('Effetto della temperatura e della penalità di ripetizione\n'
             'sulla distribuzione softmax', fontsize=13)
plt.tight_layout()
plt.show()

#### Come cambia la scelta dell'agente

In [ ]:
def serve_meal_sampled(agent, temperature=1.0, delay=1.5):
    """
    Use the trained agent to compose a full meal step by step using softmax sampling.
    Prints each choice with a delay to simulate a narrative animation.
    """
    course_idx       = 0
    used_ingredients = set()
    menu             = []
    total_reward     = 0

    print("=" * 55)
    print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
    print(f"       temperatura τ = {temperature}")
    print("=" * 55)
    time.sleep(delay)

    while not is_terminal(course_idx):
        state    = make_state(course_idx, used_ingredients)
        q_values = agent._get_q(state)
        probs    = softmax(q_values, temperature=temperature)
        action   = np.random.choice(N_ACTIONS, p=probs)
        reward   = get_reward(course_idx, action, used_ingredients)

        # Show deliberation
        clear_output(wait=True)
        print("=" * 55)
        print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
        print(f"       temperatura τ = {temperature}")
        print("=" * 55)
        for past_course, past_action in zip(COURSES[:course_idx], menu):
            print(f"\n  {past_course:<12} →  {DISHES[past_action]}")
        print(f"\n  {COURSES[course_idx]:<12} →  ...")
        print(f"\n  Il Cuoco considera:")
        for i, (dish, q, p) in enumerate(zip(DISHES, q_values, probs)):
            bar    = '█' * int(p * 20)
            marker = " ← scelta" if i == action else ""
            print(f"  {dish:<25} Q={q:+.2f}  p={p:.2f} {bar}{marker}")

        time.sleep(delay)

        # Show choice
        clear_output(wait=True)
        print("=" * 55)
        print("       🍽️  IL CUOCO COMPONE IL MENU  🍽️")
        print(f"       temperatura τ = {temperature}")
        print("=" * 55)
        for past_course, past_action in zip(COURSES[:course_idx], menu):
            print(f"\n  {past_course:<12} →  {DISHES[past_action]}")
        print(f"\n  {COURSES[course_idx]:<12} →  {DISHES[action]}  (reward: {reward:+.1f})")

        used_ingredients.add(MAIN_INGREDIENTS[action])
        menu.append(action)
        total_reward += reward
        course_idx += 1

        time.sleep(delay)

    # Final summary
    clear_output(wait=True)
    print("=" * 55)
    print("       🍽️  IL MENU È PRONTO  🍽️")
    print("=" * 55)
    for course, action in zip(COURSES, menu):
        print(f"\n  {course:<12} →  {DISHES[action]}")
    print(f"\n  Reward totale: {total_reward:+.1f}")
    print("=" * 55)

# Confronto diretto tra due temperature
serve_meal_sampled(agent, temperature=0.1, delay=1.5)   # near-greedy
serve_meal_sampled(agent, temperature=5.0, delay=1.5)   # creative

Confronto sulla generazione di menu diversi a partire da diverse temperature

In [ ]:
def generate_menu(agent, temperature):
    """Generate a full menu by sampling from softmax at each course."""
    course_idx       = 0
    used_ingredients = set()
    menu             = []
    total_reward     = 0

    while not is_terminal(course_idx):
        state  = make_state(course_idx, used_ingredients)
        probs  = softmax(agent._get_q(state), temperature=temperature)
        action = np.random.choice(N_ACTIONS, p=probs)
        reward = get_reward(course_idx, action, used_ingredients)

        used_ingredients.add(MAIN_INGREDIENTS[action])
        menu.append(action)
        total_reward += reward
        course_idx += 1

    return menu, total_reward

def print_menus(agent, n=10, temperature=1.0):
    """Generate and print n menus compactly with total reward."""
    print(f"\nTemperatura τ = {temperature}")
    print("-" * 75)
    for i in range(n):
        menu, total_reward = generate_menu(agent, temperature)
        dishes = "  |  ".join(f"{COURSES[c]}: {DISHES[a]:<22}" for c, a in enumerate(menu))
        print(f"Menu {i+1:>2}  {dishes}  reward: {total_reward:+.1f}")
    print("-" * 75)

# Low temperature: near-greedy, consistent menus
print_menus(agent, n=10, temperature=0.1)

# High temperature: creative, varied menus
print_menus(agent, n=10, temperature=0.9)